In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
train_df = pd.read_csv("data/train.csv",encoding="utf-8")
test_df = pd.read_csv("data/test.csv",encoding="utf-8")

In [3]:
import os
print(os.getcwd())

C:\Users\Berkay\PycharmProjects\sandbox\titanic_comp


In [4]:
train_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
train_df.count()

PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64

In [6]:
train_df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
title_map = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mrs": "Mrs",
    "Mme": "Mrs",
    "Master": "Master",
    "Rev": "Rare",
    "Dr": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Don": "Rare",
    "Dona": "Rare",
    "Lady": "Rare",
    "Sir": "Rare",
    "Capt": "Rare",
    "the Countess": "Rare",
    "Jonkheer": "Rare",
}

In [8]:
def preprocess_df(df):

    drop_cols = ["PassengerId","Name","Ticket",]

    #get title
    get_name = df["Name"].apply(lambda x: x.split(",")[1])
    get_title = get_name.apply(lambda x: x.split(".")[0])
    df["Title"] = get_title
    df["Title"] = df["Title"].str.strip().map(title_map)

    #get Cabin title
    cabin_list = df["Cabin"].astype(str).to_list()
    cabins = []
    for x in cabin_list:
        if str(x) == "nan":
            cabins.append("Unknown")
        else:
            cabins.append(x[0])
    df["Cabin_Title"] = cabins

    #Embarked
    df["Embarked"] = df["Embarked"].fillna("S") ##fill embarked na with S as its the most common one

    #Gender
    df["Sex"] = df["Sex"].map({"male":1, "female":0}) ## map sex col to 0 and 1

    df['TicketCount'] = df.groupby('Ticket')['Ticket'].transform('count') ##counts how many people holding the same ticket
    df = df.drop(columns='Ticket') ##drops Ticket after
    df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True) ##turn the columns into one hot

    return df


In [9]:
train_pro_df = preprocess_df(train_df)

In [10]:
train_df.groupby(["Sex","Embarked"]).count()

PassengerId  Survived  Pclass  Name  Age  SibSp  Parch  Ticket  \
Sex Embarked                                                                   
0   C                  73        73      73    73   61     73     73      73   
    Q                  36        36      36    36   12     36     36      36   
    S                 205       205     205   205  188    205    205     205   
1   C                  95        95      95    95   69     95     95      95   
    Q                  41        41      41    41   16     41     41      41   
    S                 441       441     441   441  368    441    441     441   

              Fare  Cabin  Title  Cabin_Title  TicketCount  
Sex Embarked                                                
0   C           73     37     73           73           73  
    Q           36      2     36           36           36  
    S          205     58    205          205          205  
1   C           95     32     95           95           95  
    Q           41      2     41           41           41  
    S          441     73    441          441          441

In [11]:
train_df.iloc[train_df["Embarked"].isna()]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Cabin_Title,TicketCount


In [12]:
train_pro_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Cabin          687
Cabin_Title      0
TicketCount      0
Embarked_Q       0
Embarked_S       0
Title_Miss       0
Title_Mr         0
Title_Mrs        0
Title_Rare       0
dtype: int64

In [13]:
res_cabin = train_pro_df.drop(["Cabin","Cabin_Title","Name","PassengerId"], axis=1)

In [14]:
res_cabin.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,TicketCount,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,1,22.0,1,0,7.2500,1,False,True,False,True,False,False
1,1,1,0,38.0,1,0,71.2833,1,False,False,False,False,True,False
2,1,3,0,26.0,0,0,7.9250,1,False,True,True,False,False,False
3,1,1,0,35.0,1,0,53.1000,2,False,True,False,False,True,False
4,0,3,1,35.0,0,0,8.0500,1,False,True,False,True,False,False
